### imports

In [1]:
import requests
from lxml import etree

### example ids

In [2]:
# examples ACs:
# AC02724373
# AC10790622

# example BCs:
# +Z202920203
# +Z240852802

### AC -> InventoryInfo + BC

In [2]:
def getInfoInventoryBCs(AC):
    # sru marcxml

    res1 = requests.get(f"https://obv-at-oenb.alma.exlibrisgroup.com/view/sru/43ACC_ONB?version=1.2&query=alma.local_control_field_009={AC}&operation=searchRetrieve")
    res1.raise_for_status()


    # get mms-id from response xml

    ns = {"srw": "http://www.loc.gov/zing/srw/", "marc": "http://www.loc.gov/MARC21/slim", "xb": "http://www.exlibris.com/repository/search/xmlbeans/"}

    res1Root = etree.fromstring(res1.content)
    mmsID = res1Root.find('srw:records/srw:record/srw:recordData/marc:record/marc:controlfield[@tag="001"]', namespaces=ns).text

    avaList = res1Root.findall('srw:records/srw:record/srw:recordData/marc:record/marc:datafield[@tag="AVA"]', namespaces=ns)
    libLocCNList = []

    for avaElem in avaList:
        lib = avaElem.find('marc:subfield[@code="b"]', namespaces=ns).text
        loc = avaElem.find('marc:subfield[@code="j"]', namespaces=ns).text
        cn = avaElem.find('marc:subfield[@code="d"]', namespaces=ns).text

        libLocCNList.append(cn + " " + lib + " " + loc)


    # sru(? probably) rda/rdf

    res2 = requests.get(f"https://open-na.hosted.exlibrisgroup.com/alma/43ACC_ONB/rda/entity/manifestation/{mmsID}.rdf")
    res2.raise_for_status()

    res2Root = etree.fromstring(res2.content)

    ns = {
        "madsrdfs": "http://www.loc.gov/mads/rdf/v1#",
        "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
        "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "rdac": "http://rdaregistry.info/Elements/c/",
        "rdaw": "http://rdaregistry.info/Elements/w/",
        "rdae": "http://rdaregistry.info/Elements/e/",
        "rdam": "http://rdaregistry.info/Elements/m/",
        "rdai": "http://rdaregistry.info/Elements/i/",
        "rdaa": "http://rdaregistry.info/Elements/a/"
    }

    bcElemList = res2Root.findall("rdac:Manifestation/rdam:relatedItem", namespaces=ns)

    bcList = [item.text for item in bcElemList]

    return libLocCNList, bcList

In [4]:
getInfoInventoryBCs("AC10790622")

(['309857-D.Adl.6 ZALT MAG'], ['+Z160265108'])

### BC -> InventoryInfo + AC + MMS-ID

In [5]:
def getInfoInventoryACMMSID(BC):
    # sru marcxml

    res = requests.get(f"https://obv-at-oenb.alma.exlibrisgroup.com/view/sru/43ACC_ONB?version=1.2&query=alma.barcode={BC}&maximumRecords=1&operation=searchRetrieve")
    res.raise_for_status()


    # get mms-id from response xml

    ns = {"srw": "http://www.loc.gov/zing/srw/", "marc": "http://www.loc.gov/MARC21/slim"}

    resRoot = etree.fromstring(res.content)
    mmsID = resRoot.find('srw:records/srw:record/srw:recordData/marc:record/marc:controlfield[@tag="001"]', namespaces=ns).text
    ac = resRoot.find('srw:records/srw:record/srw:recordData/marc:record/marc:controlfield[@tag="009"]', namespaces=ns).text

    avaList = resRoot.findall('srw:records/srw:record/srw:recordData/marc:record/marc:datafield[@tag="AVA"]', namespaces=ns)
    libLocCNList = []

    for avaElem in avaList:
        lib = avaElem.find('marc:subfield[@code="b"]', namespaces=ns).text
        loc = avaElem.find('marc:subfield[@code="j"]', namespaces=ns).text
        cn = avaElem.find('marc:subfield[@code="d"]', namespaces=ns).text

        libLocCNList.append(cn + " " + lib + " " + loc)

    return libLocCNList, ac, mmsID

In [6]:
getInfoInventoryACMMSID("+Z240852802")

(['394926-C ZNEU PER', '386346-C ZFID MAG'],
 'AC02724373',
 '990008083740603338')